In [ ]:
import psycopg2
import os
import tempfile

def get_cert_file_from_env():
    cert_content = os.environ.get('IBM_DB_CERTIFICATE')
    if not cert_content:
        return None
    
    # Replace \n with actual newlines
    cert_content = cert_content.replace('\\n', '\n')
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.crt', delete=False) as f:
        f.write(cert_content)
        return f.name

try:
    cert_file = get_cert_file_from_env()
    
    db_config = {
        'host': '77ffc5dd-8640-4646-a7a6-beb1dea99edb.bn2a2uid0up8mv7mv2ig.databases.appdomain.cloud',
        'port': 31173,
        'database': 'ibmclouddb',
        'user': os.environ.get('IBM_DB_USER'),
        'password': os.environ.get('IBM_DB_PASSWORD'),
        'sslmode': 'require',
        'sslrootcert': cert_file,
        'connect_timeout': 15
    }
    
    conn = psycopg2.connect(**db_config)
    cursor = conn.cursor()
    print("Connected successfully!")
    
except Exception as e:
    print(f"Error: {str(e)}")

In [5]:
import os
print(os.environ.get('IBM_DB_USER'))

None


In [ ]:


analysis_queries = [
    """
    -- Top 5 OEMs by total vehicles
    SELECT 
        oem,
        SUM(vehicle_count) as total_vehicles,
        COUNT(DISTINCT level_0_country) as countries,
        COUNT(DISTINCT body_type) as body_types
    FROM fact_registered_vehicles
    GROUP BY oem
    ORDER BY total_vehicles DESC
    LIMIT 5;
    """,
    
    """
    -- Market share by country and body type
    SELECT 
        level_0_country,
        body_type,
        COUNT(DISTINCT oem) as num_oems,
        SUM(total_vehicles_country) as total_vehicles,
        AVG(market_share_country) as avg_market_share
    FROM fact_market_share_country
    GROUP BY level_0_country, body_type
    ORDER BY total_vehicles DESC
    LIMIT 5;
    """,
    
    """
    -- Fuel type distribution
    SELECT 
        fuel_type,
        "Energy_Source",
        COUNT(DISTINCT oem) as num_oems,
        SUM(vehicle_count) as total_vehicles,
        COUNT(DISTINCT level_0_country) as num_countries
    FROM fact_registered_vehicles
    GROUP BY fuel_type, "Energy_Source"
    ORDER BY total_vehicles DESC
    LIMIT 5;
    """,
    
    """
    -- Regional market share analysis
    SELECT 
        level_0_country,
        level_1_region_name,
        COUNT(DISTINCT oem) as num_oems,
        AVG(market_share_district) as avg_market_share,
        SUM(total_vehicles_district) as total_vehicles
    FROM fact_market_share_district
    GROUP BY level_0_country, level_1_region_name
    ORDER BY total_vehicles DESC
    LIMIT 5;
    """
]


try: 
    # Run analysis queries
    print("\nRunning analysis queries...")
    for i, query in enumerate(analysis_queries, 1):
        print(f"\nAnalysis Query {i}:")
        print("-" * 80)
        cursor.execute(query)
        columns = [desc[0] for desc in cursor.description]
        print("Columns:", columns)
        print("\nResults:")
        results = cursor.fetchall()
        for row in results:
            print(row)
                
except Exception as e:
    print(f"Error: {str(e)}")
finally:
    if cursor:
        cursor.close()
    if conn:
        conn.close()

print("\nAnalysis completed.")